# دفتر ملاحظات التحليل الإحصائي المتقدم\n\n**إعداد الدكتور: حاسم أحمد الجزار**\n\nهذا الدفتر يوفر طريقة تفاعلية لإجراء التحليلات الإحصائية المتقدمة. يرجى تشغيل الخلايا بالترتيب.\n\n**تعليمات:**\n1. قم بتشغيل كل خلية رمادية بالضغط على زر التشغيل (▶) الذي يظهر على اليسار.\n2. اتبع التعليمات في كل قسم.

### الخطوة 1: تثبيت المكتبات المطلوبة

In [ ]:
!pip install pandas openpyxl statsmodels factor_analyzer scikit-learn==1.4.2

### الخطوة 2: تحميل ملف البيانات\n\nقم بتشغيل الخلية التالية، ثم انقر فوق الزر **Choose Files** لاختيار ملف البيانات (CSV أو Excel) من جهاز الكمبيوتر الخاص بك.

In [ ]:
import pandas as pd\nimport io\nfrom google.colab import files\n\ndef upload_and_load_data():\n    uploaded = files.upload()\n    if not uploaded:\n        print("لم يتم تحميل أي ملف.")\n        return None\n    \n    filename = list(uploaded.keys())[0]\n    \n    try:\n        if filename.endswith('.csv'):\n            df = pd.read_csv(io.BytesIO(uploaded[filename]))\n        elif filename.endswith(('.xls', '.xlsx')):\n            df = pd.read_excel(io.BytesIO(uploaded[filename]))\n        else:\n            print(f"نوع الملف '{filename.split('.')[-1]}' غير مدعوم. يرجى تحميل CSV أو Excel.")\n            return None\n        \n        print(f"تم تحميل ومعالجة ملف '{filename}' بنجاح.")\n        # Convert all possible columns to numeric\n        for col in df.columns:\n            df[col] = pd.to_numeric(df[col], errors='ignore')\n        return df\n    except Exception as e:\n        print(f"حدث خطأ أثناء قراءة الملف: {e}")\n        return None\n\ndf = upload_and_load_data()\n\nif df is not None:\n    print("\n**معاينة البيانات (أول 5 صفوف):**")\n    display(df.head())

---

## القسم 1: الإحصاءات الوصفية\n\nتقوم الخلية التالية بحساب الإحصاءات الوصفية الأساسية (المتوسط، الانحراف المعياري، الالتواء، التفلطح) لجميع الأعمدة الرقمية في بياناتك.

In [ ]:
if df is not None:\n    # Select only numeric columns for descriptive stats\n    numeric_df = df.select_dtypes(include='number')\n    \n    if not numeric_df.empty:\n        desc_stats = pd.DataFrame({\n            'المتوسط': numeric_df.mean(),\n            'الانحراف المعياري': numeric_df.std(),\n            'الالتواء': numeric_df.skew(),\n            'التفلطح': numeric_df.kurtosis()\n        })\n        print("**نتائج الإحصاءات الوصفية:**")\n        display(desc_stats.round(3))\n    else:\n        print("لم يتم العثور على أعمدة رقمية لإجراء التحليل.")\nelse:\n    print("يرجى تحميل البيانات أولاً في الخطوة 2.")

---

## القسم 2: تحليل الانحدار المتعدد\n\n1. قم بتعديل القوائم `independent_vars` و `dependent_var` أدناه.\n2. ضع أسماء المتغيرات المستقلة في قائمة `independent_vars`.\n3. ضع اسم المتغير التابع في `dependent_var`.

In [ ]:
import statsmodels.api as sm\n\nif df is not None:\n    # --- قم بالتعديل هنا --- \n    independent_vars = ['V1', 'V2', 'V3'] # مثال: ['var1', 'var2', 'var3']\n    dependent_var = 'Dependent' # مثال: 'dependent_var'\n    # ---------------------\n    \n    try:\n        df_regression = df.dropna(subset=independent_vars + [dependent_var])\n        X = df_regression[independent_vars]\n        y = df_regression[dependent_var]\n        X = sm.add_constant(X) # Add a constant (intercept)\n\n        model = sm.OLS(y, X).fit()\n\n        print("**نتائج تحليل الانحدار:**")\n        display(model.summary())\n        \n    except KeyError as e:\n        print(f"خطأ: لم يتم العثور على المتغير {e}. يرجى التحقق من أسماء المتغيرات.")\n    except Exception as e:\n        print(f"حدث خطأ أثناء تحليل الانحدار: {e}")\nelse:\n    print("يرجى تحميل البيانات أولاً في الخطوة 2.")

---

## القسم 3: التحليل العاملي\n\n1. قم بتعديل قائمة `factor_vars` أدناه.\n2. ضع أسماء جميع المتغيرات التي تريد تضمينها في التحليل العاملي.

In [ ]:
from factor_analyzer import FactorAnalyzer\nfrom factor_analyzer.factor_analyzer import calculate_bartlett_sphericity, calculate_kmo\nimport numpy as np\n\nif df is not None:\n    # --- قم بالتعديل هنا --- \n    factor_vars = ['V1', 'V2', 'V3', 'V4', 'V5', 'V6'] # مثال: ['item1', 'item2', 'item3', ...]\n    # ---------------------\n    \n    try:\n        df_fa = df.dropna(subset=factor_vars)\n        df_selected = df_fa[factor_vars]\n\n        if df_selected.shape[0] < df_selected.shape[1] + 1:\n             print('خطأ: لا توجد بيانات كافية بعد إزالة القيم المفقودة.')\n        else:\n            # 1. Adequacy Test (KMO and Bartlett)\n            print("**1. اختبارات كفاية العينة:**")\n            kmo_all, kmo_model = calculate_kmo(df_selected)\n            print(f"Kaiser-Meyer-Olkin (KMO) Measure: {kmo_model:.4f}")\n            chi_square_value, p_value = calculate_bartlett_sphericity(df_selected)\n            print(f"Bartlett's Test of Sphericity: Chi-squared = {chi_square_value:.4f}, p-value = {p_value:.4f}\n")\n\n            # 2. Eigenvalues and Variance Explained\n            print("**2. القيم الذاتية (Eigenvalues) والتباين المفسر:**")\n            fa = FactorAnalyzer(rotation=None, n_factors=df_selected.shape[1])\n            fa.fit(df_selected)\n            ev, v = fa.get_eigenvalues()\n            eigen_df = pd.DataFrame({\n                'Eigenvalue': ev,\n                '% of Variance': (ev / np.sum(ev)) * 100,\n                'Cumulative %': ((ev / np.sum(ev)) * 100).cumsum()\n            })\n            display(eigen_df.round(3))\n\n            # 3. Factor Loadings\n            print("**3. تشبعات العوامل (Factor Loadings):**")\n            n_factors = sum(1 for i in ev if i > 1) # Number of factors with eigenvalue > 1\n            if n_factors == 0: n_factors = 1\n            fa = FactorAnalyzer(rotation="varimax", n_factors=n_factors)\n            fa.fit(df_selected)\n            loadings_df = pd.DataFrame(fa.loadings_, index=df_selected.columns)\n            display(loadings_df.round(3))\n            \n    except KeyError as e:\n        print(f"خطأ: لم يتم العثور على المتغير {e}. يرجى التحقق من أسماء المتغيرات.")\n    except Exception as e:\n        print(f"حدث خطأ أثناء التحليل العاملي: {e}")\nelse:\n    print("يرجى تحميل البيانات أولاً في الخطوة 2.")

---\n**أسألكم الدعاء لوالدتي بالرحمة والمغفرة.**